In [8]:
from keras.src.losses import mean_squared_error
from sqlalchemy.dialects.mssql.information_schema import columns
from torch.backends.opt_einsum import strategy
%load_ext autoreload
%autoreload 2

import utils
from feature_engineering import FeatureEngineering
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.types import Integer, Float, DateTime, REAL

import model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import feature_engineering
import config  # все пути и настройки проекта лежат в config.py

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ДАННЫЕ ДЛЯ ОБУЧЕНИЯ

In [9]:
np.random.seed(config.RANDOM_SEED)

users = np.random.choice(a=range(1, 140000), size=10000, replace=False)
data = pd.read_parquet(config.PARQUET_DIR,
                       filters=[('user_id', 'in', users)])

data.sort_values(by='ts', inplace=True)

X, y = data.drop(columns=['rating']), data['rating']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.75, shuffle=False)
X_valid, X_test, y_valid, y_test = train_test_split(X_valid, y_valid, train_size=0.5, shuffle=False)

ОБУЧЕНИЕ БАЗОВОЙ МОДЕЛИ

In [4]:
import model

base_model = model.Model(
    model=model.BaseModel,
    strategy='mean',
    config_value=4.0
).fit(X_train, y_train)

print(base_model.get_metrics(X_train, y_train))
print(base_model.get_metrics(X_valid, y_valid))
print(base_model.get_metrics(X_test, y_test))

{'MSE': 0.9403315599736105, 'MAE': 0.762138985362344, 'MAPE': 0.3284829795543505}
{'MSE': 1.201139382038556, 'MAE': 0.7987418685244294, 'MAPE': 0.4271952952438355}
{'MSE': 1.156006444079518, 'MAE': 0.7764144892980502, 'MAPE': 0.40494581449095735}


ОБУЧЕНИЕ БУСТИНГА (CATBOOST)

In [10]:
import catboost

cat_model = model.Model(
    model=catboost.CatBoostRegressor,
    cat_features=['month', 'hour'],
    iterations=25,
    verbose=False,
    learning_rate=0.1,
    depth=7
).fit(X_train[config.FEATS_CATBOOST], y_train)

print(cat_model.get_metrics(X_train[config.FEATS_CATBOOST], y_train))
print(cat_model.get_metrics(X_valid[config.FEATS_CATBOOST], y_valid))
print(cat_model.get_metrics(X_test[config.FEATS_CATBOOST], y_test))


{'MSE': 0.780088808783664, 'MAE': 0.6881724009960496, 'MAPE': 0.2943816511816415}
{'MSE': 0.7349708919017053, 'MAE': 0.6565325186596256, 'MAPE': 0.3061521491697829}
{'MSE': 0.7396555854367493, 'MAE': 0.6505757357111909, 'MAPE': 0.30206110469356157}


In [ ]:
# from sklearn.metrics import mean_squared_error as MSE
# from catboost import CatBoostRegressor
#
# tun = model.Tuner(
#     n_trials=100,
#     direction='minimize',
#     metric=MSE,
#     model=CatBoostRegressor
# )
#
# tun.tuning(x_train=X_train, y_train=y_train, x_valid=X_valid, y_valid=y_valid, save_mode=True)